In [ ]:
import sys

import os
import glob

import h5py

import numpy as np

from astropy.io import fits
import astropy.units as u

import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
def compare():

    # Establish common directories
    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'
    results = f'{home}/results'

    catalog = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog)

    ids_endsley2024 = hdul_endsley2024[1].data['ID']

    # Set the prefixes of the two sets of fits that this work made (one with and one without Lya but otherwise identical)
    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    m_uv_bins = ['bright', 'faint', 'vfaint']

    # Set the labels for the legends of the figure
    labels_markers = ['with Lya', 'no Lya']

    # Set the colors of the markers for the two sets of fits
    colors = ['orange', 'blue']

    fig, ax = plt.subplots()
    fig_err, ax_err = plt.subplots()

    for i, name in enumerate(names):

        for j, m_uv_bin in enumerate(m_uv_bins):

            with h5py.File(f'{results}/ew/{name}_ews_{m_uv_bin}.h5', 'r') as f:

                for k, id in enumerate(list(f.keys())):

                    ew_posterior_me = f[id]['h_beta_ews'][:] + f[id]['o_iii_ews'][:]
                    probs = f[id]['probabilities'][:]

                    ew_me, ew_lower, ew_upper = weighted_quantile(ew_posterior_me, probs, [0.5, 0.16, 0.84])

                    ew_endsley2024, ew_endsley2024_lower, ew_endsley2024_upper = hdul_endsley2024[1].data['OIIIHbEw'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_l16'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_u84'][ids_endsley2024 == id][0]

                    ax.errorbar(ew_me, ew_endsley2024, xerr=[[ew_me - ew_lower], [ew_upper - ew_me]], yerr=[[ew_endsley2024 - ew_endsley2024_lower], [ew_endsley2024_upper - ew_endsley2024]], color=colors[i], label=f'{labels_markers[i] if j == 0 and k == 0 else ''}', alpha=0.5)
                
                    ax_err.scatter(ew_upper - ew_lower, ew_endsley2024_upper - ew_endsley2024_lower, color=colors[i], label=f'{labels_markers[i] if j == 0 and k == 0 else ''}', alpha=0.5)
                
    ax.set_xlabel('log$_{10}$(EW([O III] + H$\\beta$) / $\AA$) (this work)')
    ax.set_ylabel('log$_{10}$(EW([O III] + H$\\beta$) / $\AA$) (E24)')

    ax_err.set_xlabel('log$_{10}$((P$_{84}$(EW) - P$_{16}$(EW)) / $\AA$) (this work)')
    ax_err.set_ylabel('log$_{10}$((P$_{84}$(EW) - P$_{16}$(EW)) / $\AA$) (E24)')

    ax.legend(loc='upper left')
    ax_err.legend(loc='upper left')

    # Create a one-to-one line on the figures
    ax.axline((1000,1000), slope=1, color='red', linestyle='dashed')
    ax_err.axline((1000,1000), slope=1, color='red', linestyle='dashed')

    ax.loglog()
    ax_err.loglog()

    fig.savefig(f'{figs}/compare_ew.png', dpi=200, bbox_inches='tight')
    fig_err.savefig(f'{figs}/compare_ew_errors.png', dpi=200, bbox_inches='tight')

def plot_ews():

    home = os.getcwd()
    results = f'{home}/results'

    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    for i, name in enumerate(names):

        files = glob.glob(f'{results}/beagle_fits/{name}/*_GOODS*_BEAGLE.fits.gz')

        for j, file in enumerate(files):

            id = os.path.basename(file).split('.')[0][:-7]

            hdul = fits.open(file)

            w_rest = hdul['FULL SED WL'].data['WL'][0] * u.angstrom

            seds = hdul['FULL SED'].data * u.erg / u.s / u.cm**2 / u.angstrom

            probabilities = hdul['POSTERIOR PDF'].data['probability']

            # Make an array of observed wavelengths of the model SEDs, depending on the redshift. The subok=True 
            # parameter is necessary to preserve the astropy.Quantity subclass of the wavelength array.
            #w_obs = np.broadcast_to(w_rest, (len(zs), len(w_rest)), subok=True) * (1 + zs)[:,None]

            # Place the flux densities in the observed frame
            #seds = seds / (1 + zs)[:,None]

            #dw = np.zeros(len(w_rest)) * u.angstrom
            #for j in range(len(w_rest) - 2): dw[j+1] = (w_rest[j+2] - w_rest[j]) / 2

            #hb_ews = np.zeros(len(seds)) * u.angstrom
            #oiii_ews = np.zeros(len(seds)) * u.angstrom

            ews = np.zeros((len(lines), len(seds)), dtype=np.float64) * u.angstrom
            conts = np.zeros((len(lines), len(seds)), dtype=np.float64) * u.erg / u.s / u.cm**2 / u.angstrom

            for j, sed in enumerate(seds):

                mask = np.zeros(len(w_rest), dtype=bool)

                for k, line in enumerate(lines):

                    bands_cont = lines[line][2]

                    mask_cont = np.zeros(len(w_rest), dtype=bool)

                    for band in bands_cont:
                        mask_cont |= ((w_rest >= band[0] * u.angstrom) & (w_rest <= band[1] * u.angstrom))

                    # calculate underlying continuum
                    #line_cont_idx = np.where(((w_rest >= 4775 * u.angstrom) & (w_rest <= 4825 * u.angstrom)) | ((w_rest >= 4880 * u.angstrom) & (w_rest <= 4930 * u.angstrom)))[0]
                    line_cont = np.median(sed[mask_cont])
                    conts[k,j] = line_cont
    
                    bands_line = lines[line][1]

                    mask_line = np.zeros(len(w_rest), dtype=bool)

                    for band in bands_line:
                        mask_line |= ((w_rest >= band[0] * u.angstrom) & (w_rest <= band[1] * u.angstrom))

                    # calculate H-beta flux
                    line_idx = np.where((w_rest >= 4856 * u.angstrom) & (w_rest <= 4866 * u.angstrom))[0]
                    #line_flux = np.sum(sed[line_idx] * dw[line_idx])
    
                    # calculate H-beta EW
                    #hb_ews[j] = hb_flux / hb_cont# * (1 + np.median(zs))

                    #line_ews[j] = -1 * np.trapz(1 - (sed / line_cont)[mask_line], w_rest[mask_line], axis=0)
                    ews[k,j] = -1 * np.trapz(1 - (sed / line_cont)[mask_line], w_rest[mask_line], axis=0)
                    #dw_obs = dw * (1 + zs[j])

                    #dw = np.zeros(len(w_obs[j])) * u.angstrom
                    #for k in range(len(w_obs[j]) - 2): dw[k+1] = (w_obs[j][k+2] - w_obs[j][k]) / 2

            # If the plot flag is set, make diagnostic plots of the EW calculations for each line
            if plot:

                # For each line
                for j, line in enumerate(lines):

                    # Instantiate a diagnostic figure for the EW calculations
                    fig, ax = plt.subplots()

                    # Calculate the extrema wavelengths between the line and continuum bands
                    w_min = np.min([min(lines[line][1]), min(lines[line][2])])
                    w_max = np.max([max(lines[line][1]), max(lines[line][2])])

                    # Calculate the width of the plot (before padding) based on the minimum and maximum wavelengths of the line and continuum bands
                    width = w_max - w_min #np.max([max(lines[line][1]), max(lines[line][2])]) - np.min([min(lines[line][1]), min(lines[line][2])])

                    # Create a mask to plot only the region around the line and continuum bands, with some padding on either side
                    mask = (w_rest >= (w_min - 0.1 * width) * u.angstrom) & (w_rest <= (w_max + 0.1 * width) * u.angstrom)

                    # Plot the median flux density of the model SEDs at each wavelength in the masked region
                    ax.plot(w_rest[mask], np.median(seds, axis=0)[mask], c='red', alpha=0.5)

                    # Needs to be probability weighted percentiles, not unweighted
                    #ax.fill_between(w_rest.value[mask], np.percentile(seds.value, 16, axis=0)[mask], np.percentile(seds.value, 84, axis=0)[mask], color='red', alpha=0.2)

                    # Label the axes
                    ax.set_xlabel(r'Rest wavelength ($\mathrm{\AA}$)')
                    ax.set_ylabel('Rest-frame flux density (erg s$^{-1}$ cm$^{-2}$ $\mathrm{\AA}^{-1}$)')

                    # Set the title of the plot with the BEAGLE ID
                    ax.set_title(f'{os.path.basename(file).split('_BEAGLE')[0]}')

                    # Annotate the figure with the name of the line
                    at = AnchoredText(rf'{lines[line][0]}', loc='upper right', frameon=False)
                    ax.add_artist(at)

                    ax.set_xlim(w_min - 0.05 * width, w_max + 0.05 * width)

                    # Plot each continuum-sampling band
                    for k, band in enumerate(lines[line][2]):

                        label = 'Continuum band' if k == 0 else None

                        ax.axvspan(band[0], band[1], alpha=0.2, color='blue', label=label)

                    # Plot each line-sampling band
                    for k, band in enumerate(lines[line][1]):

                        label = 'Line band' if k == 0 else None

                        ax.axvspan(band[0], band[1], alpha=0.2, color='orange', label=label)

                    ax.axhline(np.median(conts.value[j]), color='black', ls='dashed')
                    ax.fill_between(w_rest.value[mask], np.percentile(conts.value[j], 16), np.percentile(conts.value[j], 84), color='black', alpha=0.2)

                    ax.legend(loc='upper left')

                    ax.loglog()

                    plt.show()
                    plt.close('all')

def low_ew_ids():

    '''
    Print the object IDs of the sources with low EWs (< 100 angstrom) in E24, which nearly universally have much higher EWs in this work
    '''

    # Establish common directories
    home = os.getcwd()
    data = f'{home}/data'
    results = f'{home}/results'

    catalog = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog)

    ids_endsley2024 = hdul_endsley2024[1].data['ID']

    # Set the prefixes of the two sets of fits that this work made (one with and one without Lya but otherwise identical)
    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    m_uv_bins = ['bright', 'faint', 'vfaint']

    # Set the labels for the legends of the figure
    labels_markers = ['with Lya', 'no Lya']

    # Set the colors of the markers for the two sets of fits
    colors = ['orange', 'blue']

    for i, name in enumerate(names):

        for j, m_uv_bin in enumerate(m_uv_bins):

            with h5py.File(f'{results}/ew/{name}_ews_{m_uv_bin}.h5', 'r') as f:

                for k, id in enumerate(list(f.keys())):

                    ew_posterior_me = f[id]['h_beta_ews'][:] + f[id]['o_iii_ews'][:]
                    probs = f[id]['probabilities'][:]

                    ew_me, ew_lower, ew_upper = weighted_quantile(ew_posterior_me, probs, [0.5, 0.16, 0.84])

                    ew_endsley2024, ew_endsley2024_lower, ew_endsley2024_upper = hdul_endsley2024[1].data['OIIIHbEw'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_l16'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_u84'][ids_endsley2024 == id][0]

In [ ]:
compare()